# 02 · Building a better spread forecast

**Francis White**

We compare eight models, adding one idea at a time: stock averages, mean reversion, other stocks’ activity, nonlinear effects and an ensemble. This shows which choices help predict the change in average spread.

The original model code and saved results are included below. Setup works locally or in Colab. These results use the original 10-fold evaluation; [the final study](04_forecasting_and_trading.ipynb) uses a later design. This extraction has not been rerun on the full dataset.

[All notebooks](RESEARCH_GUIDE.md).


In [1]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Erdos-Projects/fall-2026-short-horizon-forecasting"
DATA_DIR = os.environ.get("OBF_DATA_DIR", "")      # e.g. "/path/to/optiver-realized-volatility-prediction"
CONFIG = "configs/default.yaml"                     # "configs/smoke.yaml" for a quick 3-stock test

if IN_COLAB:
    if not Path("fall-2026-short-horizon-forecasting").exists():
        !git clone -q {REPO_URL}
    %cd fall-2026-short-horizon-forecasting
    %pip install -q -r requirements.txt
    %pip install -q -e .
    if not DATA_DIR:
        from google.colab import userdata
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
        DATA_DIR = "/content/data/optiver-realized-volatility-prediction"
        kagglehub.competition_download("optiver-realized-volatility-prediction", output_dir=DATA_DIR)
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(root)
    sys.path.insert(0, str(root / "src"))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from obforecast.config import load_config
from obforecast.data import ParquetSource, check_layout, resolve_data_dir

cfg = load_config(CONFIG)
data_dir = resolve_data_dir(DATA_DIR or None, cfg)
layout = check_layout(data_dir)
source = ParquetSource(data_dir)
DATA_DIR = data_dir
train = pd.read_csv(data_dir / "train.csv")
STOCK_ID = int(sorted(train.stock_id.unique())[0])
def load_stock(kind, stock_id):
    return source.load(kind, int(stock_id)).sort_values(
        ["time_id", "seconds_in_bucket"]).reset_index(drop=True)
book = load_stock("book", STOCK_ID)
trades = load_stock("trade", STOCK_ID)
print(f"{train.stock_id.nunique()} stocks; sample stock {STOCK_ID}")


112 stocks; sample stock 0


In [3]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## Modeling: predicting the change in the bid-ask spread

**Goal.** For every stock in a 10-minute window, use only the first 5 minutes of book data to predict

$$y = \log\left(\frac{S_2}{S_1}\right), \qquad S_h = \frac{1}{T_h}\int_h \frac{a_t - b_t}{m_t}\,dt,$$

the log ratio of the **time-weighted average relative quoted spread** in the second half (seconds 300–599) to that in the first half (seconds 0–299).

**Choices.**
- *Time-weighted*: each book snapshot holds until the next one. Snapshots arrive when the book changes, so a plain average over rows would over-weight busy periods.
- *Relative spread*: dividing by the mid price makes spreads comparable across stocks.
- *Log ratio*: the target is symmetric and tame, while the raw ratio is skewed.

**Approach.** Fit one model shared across stocks (one row per stock and window), adding features that summarize all stocks in the same window. The other stocks' data enters as inputs to a shared model rather than through a huge multi-output model. Evaluate with **10-fold `GroupKFold` on `time_id`**, so a whole window is always in the same fold. Errors are strongly correlated within a window, so a random row split would leak information.

These models use **book data only**, so they run on every stock with book data, not just `AVAILABLE_STOCKS`.bold text

**Data coverage.** This source cell uses the real partition folders under `DATA_DIR` to identify stocks with complete files.

In [4]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def missing_stocks(folder):
    """Stocks in train.csv with no book/trade parquet under folder."""
    stock_ids = pd.read_csv(folder / "train.csv", usecols=["stock_id"]).stock_id.unique()
    def has_data(kind, stock_id):
        part = folder / f"{kind}_train.parquet" / f"stock_id={stock_id}"
        return any(p.stat().st_size > 0 for p in part.rglob("*.parquet"))
    return {kind: [int(s) for s in stock_ids if not has_data(kind, s)] for kind in ["book", "trade"]}

missing = missing_stocks(DATA_DIR)
all_stocks = pd.read_csv(DATA_DIR / "train.csv", usecols=["stock_id"]).stock_id.unique()
AVAILABLE_STOCKS = sorted(int(s) for s in all_stocks
                          if s not in missing["book"] and s not in missing["trade"])
print(f"{len(AVAILABLE_STOCKS)} stocks with full data; missing: {missing}")

BOOK_STOCKS = sorted(int(s) for s in all_stocks if s not in missing["book"])
print(f"{len(BOOK_STOCKS)} stocks with book data")

112 stocks with full data; missing: {'book': [], 'trade': []}
112 stocks with book data


### Features from the first half

Each stock's book goes onto a dense `(window × 600 seconds)` grid with forward-fill, so plain means over the grid are time-weighted averages. The target is a ratio, so most features describe where the spread sits *now* relative to its first-half average, and what is pushing it:

| Group | Features |
|---|---|
| Level | `log_S1` (and, per fold, `abn` = `log_S1` minus the stock's typical value) |
| Trend within the half | spread over the last 60/30/10 s and at second 299 relative to $S_1$; first 60 s; dispersion of log spread |
| Tick constraint | average spread in multiples of the minimum spread (≈ ticks); fraction of time at the minimum; relative tick size |
| Volatility & activity | realized volatility of the weighted average price (WAP) over the half and the last 100 s; their ratio; absolute return; number of book updates |
| Depth & shape | level-1/level-2 sizes and recent change; order imbalance; level-2 spread relative to $S_1$ |

`y_skip30` is an alternative target that drops the first 30 s of the second half. It's used in 4.5 to check whether the score comes from the carry-over at the 300 s boundary.

In [5]:
H = 300  # half-window length in seconds
BOOK_COLS = ["bid_price1", "ask_price1", "bid_price2", "ask_price2",
             "bid_size1", "ask_size1", "bid_size2", "ask_size2"]


def dense_book(book):
    # time_ids and dict of [n_windows, 600] forward-filled arrays (+ 'upd' = 1 where a snapshot arrived)
    tids = np.sort(book.time_id.unique())
    row = np.searchsorted(tids, book.time_id.to_numpy())
    col = book.seconds_in_bucket.to_numpy()
    out = {}
    for c in BOOK_COLS:
        a = np.full((len(tids), 600), np.nan)
        a[row, col] = book[c].to_numpy(dtype=float)
        out[c] = pd.DataFrame(a).ffill(axis=1).to_numpy()
    upd = np.zeros((len(tids), 600))
    upd[row, col] = 1
    out["upd"] = upd
    return tids, out


def log_mean(a, lo, hi):
    return np.log(np.nanmean(a[:, lo:hi], axis=1))


def spread_features(stock_id, book):
    tids, d = dense_book(book)
    bp1, ap1, bp2, ap2 = d["bid_price1"], d["ask_price1"], d["bid_price2"], d["ask_price2"]
    bs1, as1, bs2, as2 = d["bid_size1"], d["ask_size1"], d["bid_size2"], d["ask_size2"]
    mid = (ap1 + bp1) / 2
    s = (ap1 - bp1) / mid                      # relative quoted spread
    s2 = (ap2 - bp2) / mid                     # level-2 relative spread
    wap = (bp1 * as1 + ap1 * bs1) / (bs1 + as1)
    r = np.diff(np.log(wap), axis=1, prepend=np.log(wap[:, :1]))  # 1-second log returns

    f = pd.DataFrame({"stock_id": stock_id, "time_id": tids})
    S1 = np.nanmean(s[:, :H], axis=1)
    f["y"] = np.log(np.nanmean(s[:, H:], axis=1) / S1)
    f["y_skip30"] = np.log(np.nanmean(s[:, H + 30:], axis=1) / S1)
    f["log_S1"] = ls1 = np.log(S1)

    # spread trend within the first half, relative to the half average
    f["sp_last60"] = log_mean(s, H - 60, H) - ls1
    f["sp_last30"] = log_mean(s, H - 30, H) - ls1
    f["sp_last10"] = log_mean(s, H - 10, H) - ls1
    f["sp_now"] = np.log(s[:, H - 1]) - ls1
    f["sp_first60"] = log_mean(s, 0, 60) - ls1
    f["sp_logstd"] = np.nanstd(np.log(s[:, :H]), axis=1)

    # tick constraint: absolute spread in multiples of its minimum
    abs_sp = (ap1 - bp1)[:, :H]
    tick = np.nanmin(np.where(abs_sp > 0, abs_sp, np.nan), axis=1)
    f["sp_ticks"] = np.log(np.nanmean(abs_sp, axis=1) / tick)
    f["frac_min_tick"] = np.mean(abs_sp <= tick[:, None] * 1.001, axis=1)
    f["log_rel_tick"] = np.log(tick / mid[:, H - 1])

    # volatility and activity
    f["log_rv"] = np.log(np.sqrt((r[:, 1:H] ** 2).sum(axis=1)) + 1e-8)
    f["log_rv_last100"] = np.log(np.sqrt((r[:, H - 100:H] ** 2).sum(axis=1)) + 1e-8)
    f["rv_ratio"] = f["log_rv_last100"] - f["log_rv"]
    f["abs_ret"] = np.abs(np.log(wap[:, H - 1] / wap[:, 0]))
    n_upd = d["upd"][:, :H].sum(axis=1)
    f["log_nupd"] = np.log(n_upd)
    f["upd_last60"] = np.log(d["upd"][:, H - 60:H].sum(axis=1) + 1) - np.log(n_upd / 5 + 1)

    # depth / shape of the book
    dep1 = bs1 + as1
    f["log_depth1"] = log_mean(dep1, 0, H)
    f["log_depth2"] = log_mean(bs2 + as2, 0, H)
    f["depth1_last60"] = log_mean(dep1, H - 60, H) - f["log_depth1"]
    f["imbalance"] = np.nanmean(np.abs(bs1 - as1)[:, :H] / dep1[:, :H], axis=1)
    f["imb_now"] = ((bs1 - as1) / dep1)[:, H - 1]
    f["sp2_rel"] = log_mean(s2, 0, H) - ls1
    return f


parts = []
for i, sid in enumerate(BOOK_STOCKS):
    parts.append(spread_features(sid, load_stock("book", sid)))
    if (i + 1) % 10 == 0 or i + 1 == len(BOOK_STOCKS):
        print(f"features: {i + 1}/{len(BOOK_STOCKS)} stocks")
spread_feats = pd.concat(parts, ignore_index=True)
print(spread_feats.shape, "| any NaN:", spread_feats.isna().any().any())
spread_feats.describe().T.round(4)

features: 10/112 stocks


features: 20/112 stocks


features: 30/112 stocks


features: 40/112 stocks


features: 50/112 stocks


features: 60/112 stocks


... (3 outputs left out)

features: 100/112 stocks


features: 110/112 stocks


features: 112/112 stocks
(428932, 26) | any NaN: False


,count,mean,std,min,25%,50%,75%,max
stock_id,428932.0,62.4379,37.1264,0.0000,30.0000,63.0000,96.0000,126.0000
time_id,428932.0,16038.9727,9365.1037,5.0000,7854.0000,15853.0000,23994.0000,32767.0000
y,428932.0,-0.0357,0.1757,-2.3964,-0.1229,-0.0163,0.0547,1.3903
y_skip30,428932.0,-0.0376,0.1829,-2.4709,-0.1286,-0.0171,0.0565,1.4123
log_S1,428932.0,-7.6495,0.8311,-10.5356,-8.2372,-7.6972,-7.1009,-3.9395
sp_last60,428932.0,-0.0300,0.1666,-1.4327,-0.1159,-0.0127,0.0621,0.8840
sp_last30,428932.0,-0.0387,0.2093,-1.6025,-0.1498,-0.0176,0.0793,1.1118
sp_last10,428932.0,-0.0511,0.2643,-2.2348,-0.1940,-0.0253,0.1035,1.5489
sp_now,428932.0,-0.0726,0.3455,-3.5800,-0.2621,-0.0366,0.1365,1.7710
sp_first60,428932.0,0.0096,0.1656,-1.3103,-0.0738,0.0022,0.1016,1.1589


### Train/test split by window

A random 20% of windows (`time_id`s) is held out as the final test set **before** anything looks at the target. The next cells use only the development set `dev`; `test` is for a single final score once model choices are final.

- Split by window, so every stock's row for a held-out window is in `test`. Errors and the market-wide move are shared within a window.
- The seed is fixed, so the split is identical on every run and on the full dataset. Don't change it after seeing test scores.

In [6]:
TEST_FRAC = 0.2
SPLIT_SEED = 0

tids = np.sort(spread_feats.time_id.unique())
TEST_TIDS = np.random.default_rng(SPLIT_SEED).choice(tids, size=round(TEST_FRAC * len(tids)), replace=False)
is_test = spread_feats.time_id.isin(TEST_TIDS)
dev = spread_feats[~is_test].reset_index(drop=True)
test = spread_feats[is_test].reset_index(drop=True)
print(f"dev:  {dev.time_id.nunique():,} windows, {len(dev):,} rows")
print(f"test: {test.time_id.nunique():,} windows, {len(test):,} rows")

dev:  3,064 windows, 343,144 rows
test: 766 windows, 85,788 rows


### Quick look at the target

Before modeling, three questions:
1. How much of $y$ is **shared by all stocks in a window**? This is the share of variance explained by the window mean.
2. How much is a **fixed stock effect**? This is the share explained by the stock mean.
3. Do the main signals point the same way **for every stock**? If so, one shared model is justified.

**Descriptive scope.** This original diagnostic uses all `spread_feats`, including the benchmark windows, for descriptive variance/correlation summaries. The fitting/CV cells use `dev` only. This does not create an untouched benchmark.

In [7]:
y = spread_feats.y
by_window = spread_feats.groupby("time_id").y.transform("mean")
by_stock = spread_feats.groupby("stock_id").y.transform("mean")
print(f"var(y) = {y.var():.4f}")
print(f"share of variance explained by window mean: {1 - ((y - by_window) ** 2).mean() / y.var():.3f}")
print(f"share of variance explained by stock mean:  {1 - ((y - by_stock) ** 2).mean() / y.var():.3f}")

eda = spread_feats.assign(abn=spread_feats.log_S1 - spread_feats.groupby("stock_id").log_S1.transform("mean"))
signals = ["abn", "sp_last30", "sp_last60", "sp_first60", "log_rv", "rv_ratio", "sp2_rel"]
per_stock_corr = eda.groupby("stock_id")[signals + ["y"]].corr(method="spearman").xs("y", level=1)[signals]
per_stock = eda.groupby("stock_id").agg(y_std=("y", "std"), frac_min_tick=("frac_min_tick", "mean"))
display(per_stock.join(per_stock_corr).round(2).sort_values("frac_min_tick"))
print("Fraction of stocks where each signal has the same sign as its median:")
print((np.sign(per_stock_corr) == np.sign(per_stock_corr.median())).mean().round(2).to_string())

var(y) = 0.0309
share of variance explained by window mean: 0.176
share of variance explained by stock mean:  0.003


,y_std,frac_min_tick,abn,sp_last30,sp_last60,sp_first60,log_rv,rv_ratio,sp2_rel
stock_id,,,,,,,,,
6,0.17,0.01,-0.27,0.20,0.19,-0.06,-0.19,0.15,0.22
23,0.18,0.01,-0.22,0.27,0.27,-0.12,-0.17,0.16,0.24
22,0.18,0.01,-0.28,0.25,0.24,-0.11,-0.22,0.15,0.26
17,0.17,0.01,-0.28,0.24,0.22,-0.11,-0.22,0.15,0.25
62,0.17,0.01,-0.23,0.25,0.23,-0.10,-0.18,0.17,0.23
...,...,...,...,...,...,...,...,...,...
108,0.10,0.94,-0.16,0.29,0.27,-0.09,-0.22,0.14,0.46
41,0.09,0.95,-0.25,0.41,0.34,-0.10,-0.25,0.12,0.52
77,0.05,0.95,-0.11,0.18,0.19,-0.06,-0.14,0.08,0.40


Fraction of stocks where each signal has the same sign as its median:
abn           1.00
sp_last30     1.00
sp_last60     1.00
sp_first60    0.99
log_rv        1.00
rv_ratio      1.00
sp2_rel       1.00


### Models

Each model adds one idea, so the table shows what each idea is worth:

| # | Model | What it tests |
|---|---|---|
| M0 | predict 0 | "no change" |
| M1 | per-stock mean of $y$ | stock-specific drift |
| M2 | OLS on `abn` only | pure mean reversion |
| M3 | ridge, own-stock features | linear model, full first-half profile |
| M4 | ridge + market features | same, plus other stocks in the window |
| M5 | gradient boosting (`HistGradientBoostingRegressor`), own-stock features | nonlinearity / interactions |
| M6 | gradient boosting + market features | both |
| M7 | average of M4 and M6 | ensemble |

**Avoiding leakage.**
- Features that need a stock's typical level (`abn`, `rv_abn`, `nupd_abn`, `depth_abn`) use stock means from the **training folds only**.
- Market features (`mkt_*`) are averages over stocks *in the same window*, built from first-half features only. Folds are grouped by `time_id`, so this never leaks targets.
- `rel_*` features are a stock's value minus the market average.

**Original experiment.** Market means here include the focal stock. The final study uses leave-one-out means and nested selection; its scores belong to that different specification.

In [8]:
OWN = ["log_S1", "sp_last60", "sp_last30", "sp_last10", "sp_now", "sp_first60",
       "sp_logstd", "sp_ticks", "frac_min_tick", "log_rel_tick", "log_rv",
       "log_rv_last100", "rv_ratio", "abs_ret", "log_nupd", "upd_last60",
       "log_depth1", "log_depth2", "depth1_last60", "imbalance", "imb_now", "sp2_rel"]
# feature -> source column; feature = source minus the stock's training-fold mean
ABN = {"abn": "log_S1", "rv_abn": "log_rv", "nupd_abn": "log_nupd", "depth_abn": "log_depth1"}
MKT_SRC = ["abn", "rv_abn", "nupd_abn", "depth_abn", "sp_last60", "sp_last30", "sp_now",
           "sp_first60", "rv_ratio", "upd_last60", "depth1_last60", "sp2_rel"]
OWN_F = OWN + list(ABN)
MKT_F = OWN_F + ["mkt_" + c for c in MKT_SRC] + ["rel_abn", "rel_last60", "mkt_disp_abn"]


def add_fold_features(tr, te):
    out = []
    for d in (tr, te):
        d = d.copy()
        for new, src in ABN.items():
            d[new] = d[src] - d.stock_id.map(tr.groupby("stock_id")[src].mean())
        m = d.groupby("time_id")[MKT_SRC].transform("mean")
        for c in MKT_SRC:
            d["mkt_" + c] = m[c]
        d["rel_abn"] = d.abn - d.mkt_abn
        d["rel_last60"] = d.sp_last60 - d.mkt_sp_last60
        d["mkt_disp_abn"] = d.groupby("time_id").abn.transform("std")
        out.append(d)
    return out


def ridge():
    return make_pipeline(StandardScaler(), Ridge(alpha=10.0))


def hgb():
    # stock_id (first column) is treated as categorical
    return HistGradientBoostingRegressor(
        learning_rate=0.05, max_iter=500, max_leaf_nodes=31, min_samples_leaf=200,
        l2_regularization=1.0, categorical_features=[0], random_state=0)


MODELS = {
    "M0 zero": lambda tr, te: np.zeros(len(te)),
    "M1 stock mean": lambda tr, te: te.stock_id.map(tr.groupby("stock_id").y.mean()).to_numpy(),
    "M2 OLS abn": lambda tr, te: LinearRegression().fit(tr[["abn"]], tr.y).predict(te[["abn"]]),
    "M3 ridge own": lambda tr, te: ridge().fit(tr[OWN_F], tr.y).predict(te[OWN_F]),
    "M4 ridge +mkt": lambda tr, te: ridge().fit(tr[MKT_F], tr.y).predict(te[MKT_F]),
    "M5 HGB own": lambda tr, te: hgb().fit(tr[["stock_id"] + OWN_F], tr.y).predict(te[["stock_id"] + OWN_F]),
    "M6 HGB +mkt": lambda tr, te: hgb().fit(tr[["stock_id"] + MKT_F], tr.y).predict(te[["stock_id"] + MKT_F]),
}
ENSEMBLES = {"M7 avg(M4,M6)": ["M4 ridge +mkt", "M6 HGB +mkt"]}

### 10-fold cross-validation (development set)

Cross-validation runs on `dev` only; the test windows are not touched here. Metrics are computed on the pooled out-of-fold predictions:
- **RMSE** and **R²**, where R² is relative to predicting the overall mean.
- **R² market**: R² of the window-average prediction against the window-average $y$. Can the model predict the market-wide change in liquidity?
- **R² idio**: R² on deviations from the window average. Can the model predict which stocks widen or narrow relative to the others?
- **Within-window IC**: Spearman correlation between prediction and $y$ across stocks within a window, averaged over windows.
- **R² by group**: tick-bound stocks (at the minimum spread more than half the time) against the rest. Tick-bound stocks have a nearly constant spread and can flatter the overall score.

**Evaluate the progression.** Compare each added idea within this table. The original categorical stock IDs and model hyperparameters are preserved.

In [9]:
def within_window_ic(time_id, y, p):
    d = pd.DataFrame({"t": time_id, "y": y, "p": p})
    r = d.groupby("t")[["y", "p"]].rank()
    r = r - r.groupby(d.t).transform("mean")
    num = (r.y * r.p).groupby(d.t).sum()
    den = np.sqrt((r.y ** 2).groupby(d.t).sum() * (r.p ** 2).groupby(d.t).sum())
    return (num / den.replace(0, np.nan)).mean()


def cross_validate(data, target="y", models=MODELS, ensembles=ENSEMBLES, n_splits=10):
    data = data.assign(y=data[target]).reset_index(drop=True)
    names = list(models) + list(ensembles)
    oof = pd.DataFrame(np.nan, index=data.index, columns=names)
    fold_rmse = {n: [] for n in names}
    for k, (tri, tei) in enumerate(GroupKFold(n_splits=n_splits).split(data, groups=data.time_id)):
        tr, te = add_fold_features(data.iloc[tri], data.iloc[tei])
        for n, fit_predict in models.items():
            oof.loc[tei, n] = fit_predict(tr, te)
        for n, members in ensembles.items():
            oof.loc[tei, n] = oof.loc[tei, members].mean(axis=1)
        for n in names:
            fold_rmse[n].append(np.sqrt(((oof.loc[tei, n] - te.y.to_numpy()) ** 2).mean()))
        print(f"fold {k + 1}/{n_splits} done")
    return data, oof, fold_rmse


def evaluate(data, oof, fold_rmse):
    y = data.y.to_numpy()
    t = data.time_id.to_numpy()
    tick_bound = (data.stock_id.map(data.groupby("stock_id").frac_min_tick.mean()) > 0.5).to_numpy()
    y_w = data.groupby("time_id").y.transform("mean").to_numpy()

    def r2(true, pred):
        return 1 - ((true - pred) ** 2).sum() / ((true - true.mean()) ** 2).sum()

    rows = {}
    for n in oof.columns:
        p = oof[n].to_numpy()
        p_w = pd.Series(p).groupby(t).transform("mean").to_numpy()
        win = pd.DataFrame({"y": y, "p": p}).groupby(t).mean()
        rows[n] = {
            "RMSE": np.sqrt(((y - p) ** 2).mean()),
            "RMSE fold sd": np.std(fold_rmse[n]),
            "R2": r2(y, p),
            "R2 market": r2(win.y.to_numpy(), win.p.to_numpy()),
            "R2 idio": 1 - ((y - y_w - (p - p_w)) ** 2).sum() / ((y - y_w) ** 2).sum(),
            "within-window IC": within_window_ic(t, y, p),
            "R2 tick-bound": r2(y[tick_bound], p[tick_bound]),
            "R2 wide-spread": r2(y[~tick_bound], p[~tick_bound]),
        }
    return pd.DataFrame(rows).T


cv_data, oof, fold_rmse = cross_validate(dev)
results = evaluate(cv_data, oof, fold_rmse)
print(f"{cv_data.stock_id.nunique()} stocks, {cv_data.time_id.nunique()} windows, "
      f"{(cv_data.stock_id.map(cv_data.groupby('stock_id').frac_min_tick.mean()) > 0.5).groupby(cv_data.stock_id).first().sum()} tick-bound stocks")
results.round(4)

fold 1/10 done


fold 2/10 done


fold 3/10 done


fold 4/10 done


fold 5/10 done


fold 6/10 done


... (2 outputs left out)

fold 9/10 done


fold 10/10 done


112 stocks, 3064 windows, 21 tick-bound stocks


,RMSE,RMSE fold sd,R2,R2 market,R2 idio,within-window IC,R2 tick-bound,R2 wide-spread
M0 zero,0.1792,0.0044,-0.0413,-0.2359,0.0000,NaN,-0.0449,-0.0413
M1 stock mean,0.1753,0.0041,0.0027,-0.0003,0.0034,0.0487,0.0095,0.0014
M2 OLS abn,0.1647,0.0028,0.1201,0.3174,0.0783,0.2557,0.2257,0.1070
M3 ridge own,0.1538,0.0022,0.2330,0.5295,0.1701,0.3544,0.4199,0.2101
M4 ridge +mkt,0.1478,0.0018,0.2912,0.7994,0.1834,0.3695,0.4488,0.2718
M5 HGB own,0.1474,0.0018,0.2954,0.6731,0.2153,0.3865,0.6061,0.2576
M6 HGB +mkt,0.1402,0.0015,0.3619,0.8311,0.2624,0.4300,0.6425,0.3278
"M7 avg(M4,M6)",0.1419,0.0016,0.3466,0.8277,0.2446,0.4165,0.5946,0.3164


## Connection to the final study

This progression motivates the spread target, stock-relative normalization, market inputs and nonlinear model in [the final study](04_forecasting_and_trading.ipynb). Its comparison table is valuable research evidence even though the final pipeline uses a later design. [Nhi Dinh’s widening-event classifier](03_spread_widening_events.ipynb) asks a separate binary-event question.